# Desafio de análise de crédito

Neste projeto, meu objetivo é entender melhor por que algumas propostas de crédito acabam em inadimplência e verificar se é possível usar as informações disponíveis para identificar casos com maior risco antes da aprovação.

Vou seguir uma lógica bem parecida com a que um aluno faria ao desenvolver uma análise de dados: primeiro entendo o problema, depois exploro os dados, verifico a qualidade da base, preparo as variáveis e, por fim, treino um modelo para avaliar o resultado.

A ideia é mostrar a evolução da análise como se estivesse sendo apresentada em uma reunião, explicando cada etapa e a decisão por trás de cada escolha.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

sns.set_theme(style="whitegrid")

df = pd.read_csv('dataset.csv')
print('Primeiras linhas do dataset:')
display(df.head())

In [ ]:
print('Formato do dataset:', df.shape)
print('\nTipos de dados:\n', df.dtypes)
print('\nValores ausentes:\n', df.isna().sum())
print('\nDistribuição da variável alvo:\n', df['inadimplente'].value_counts(normalize=True).round(4))

### Primeiro olhar sobre os dados

Ao executar essa parte, o objetivo é entender a estrutura da base antes de qualquer análise mais profunda. Nesse momento, eu quero responder perguntas simples: quantas linhas e colunas existem, que tipo de informação cada coluna guarda, se há valores faltantes e como a variável alvo está distribuída.

Se a distribuição da variável resposta estiver equilibrada o suficiente, a modelagem fica mais estável. Se houver forte desbalanceamento, a interpretação dos resultados precisa levar isso em conta.

## 1. Entendimento do problema

A variável que queremos prever é `inadimplente`. Ela informa se a proposta resultou em inadimplência ou não, então estamos diante de um problema de classificação binária.

A ideia é usar as informações do cliente e da proposta para prever se ele pertence ao grupo de risco de inadimplência (1) ou ao grupo sem risco (0). Isso faz sentido para uma instituição financeira, porque ajuda a analisar a probabilidade de problema antes de conceder o crédito.

Como a resposta é binária e temos variáveis explicativas para descrever o cliente e o empréstimo, a regressão logística se encaixa muito bem nesse cenário.

### Observação inicial das estatísticas

Essa etapa serve para entender a escala das variáveis e verificar se há valores muito altos, muito baixos ou comportamentos estranhos. Em um problema de crédito, isso é importante porque variáveis financeiras costumam ter grande impacto na análise.

Ao olhar essas estatísticas, eu procuro sinais de que algumas características dos clientes podem estar associadas ao risco, como renda, score, valor solicitado e quantidade de dívidas ativas.

### Comparando risco por categoria

Aqui eu verifico se categorias como estado, escolaridade e finalidade têm diferenças relevantes na taxa de inadimplência. Essa análise é útil porque pode mostrar se há agrupamentos de clientes que exigem mais atenção ou se o risco está mais relacionado a fatores financeiros do que a características socioeconômicas isoladas.

Em uma apresentação, esse é um momento interessante porque permite mostrar que, antes mesmo do modelo, já existem sinais de padrão no histórico.

In [ ]:
df.describe(include='all').T.round(2)

### Interpretação do comportamento das variáveis

Ao observar esses gráficos, eu procuro perceber se os clientes inadimplentes apresentam características distintas dos clientes sem problema. Se o score de crédito for menor, a renda mais baixa, o valor solicitado mais alto ou as dívidas mais numerosas, isso é um sinal forte de que a inadimplência está praticamente conectada ao risco financeiro.

A parte mais importante da análise exploratória não é apenas ver que há diferença, mas entender o que essa diferença quer dizer dentro do contexto do negócio.

### Relação entre variáveis e alvo

A matriz de correlação ajuda a identificar se as variáveis têm relação linear com a inadimplência e entre si. Isso é útil para perceber se há redundância, se algumas variáveis apontam na mesma direção e se o risco parece estar mais ligado a um conjunto específico de indicadores.

Na prática, isso me ajuda a decidir quais variáveis merecem foco no modelo e quais parecem ser menos informativas para o problema.

In [ ]:
for col in ['estado', 'escolaridade', 'finalidade']:
    print(f'\n### {col}')
    risco = df.groupby(col)['inadimplente'].mean().sort_values(ascending=False)
    print(risco.head(10).round(3).to_string())

### Resultados esperados nesta etapa

Se tudo estiver consistente, eu espero encontrar poucos ou nenhum valor ausente, sem duplicatas relevantes e com faixas plausíveis para as variáveis numéricas. Isso é um bom sinal de que a base é confiável o bastante para avança na modelagem.

Quando isso acontece, o foco da apresentação pode seguir para a parte mais interpretativa: entender os fatores que realmente parecem estar relacionados ao risco.

## 2. Análise exploratória

Antes de ajustar qualquer modelo, eu quero responder uma pergunta bem simples: os dados já mostram algum padrão claro de risco? Se sim, melhor ainda, porque isso indica que as variáveis disponíveis realmente têm relação com a inadimplência.

A minha hipótese inicial é que indicadores financeiros, como renda, score de crédito, valor solicitado e número de dívidas, tenham mais força do que características mais superficiais, como idade ou estado. Isso é um ponto importante, porque mostra que o problema é essencialmente financeiro e comportamental.

Vou comparar as variáveis numéricas entre os grupos inadimplentes e não inadimplentes para ver como elas se diferenciam.

### O que essa nova variável representa

A variável `renda_sobre_valor` me ajuda a medir o peso do empréstimo em relação à renda do cliente. Em vez de olhar apenas o valor solicitado em reais, eu passo a considerar se ele está proporcionalmente elevado em relação à capacidade financeira do solicitante.

Esse tipo de variável costuma ser muito útil em análise de crédito, porque reforça a lógica de que não basta observar o valor do empréstimo isoladamente; é preciso conhecer a relação com a renda e com o prazo.

### Validando a variável derivada

Ao olhar esse gráfico, eu quero verificar se o grupo inadimplente tem uma relação renda/valor mais apertada, ou seja, se está pedindo mais em comparação com sua renda. Se isso acontecer, então a feature realmente tem potencial explicativo.

Esse passo é importante porque mostra que a variável construída não foi feita só para aumentar o número de atributos, mas porque ela tem uma justificativa clara para o problema de crédito.

In [ ]:
num_cols = ['idade', 'renda_mensal', 'tempo_emprego_anos', 'score_credito', 'valor_solicitado', 'prazo_meses', 'dividas_ativas']

for col in num_cols:
    plt.figure(figsize=(6, 3))
    sns.boxplot(data=df, x='inadimplente', y=col)
    plt.title(f'{col} por inadimplência')
    plt.show()

summary = df.groupby('inadimplente')[num_cols].agg(['mean', 'median']).T
summary.round(2)

### Decisões importantes nesta etapa

Nesta parte, eu escolhi usar um pipeline para deixar o processo mais organizado e reproduzível. O pipeline faz a imputação dos valores faltantes e também transforma variáveis numéricas e categóricas antes do treinamento.

Para as variáveis numéricas, eu usei padronização, porque elas têm escalas diferentes. Para as categóricas, foi necessário transformar as categorias em valores numéricos para que o modelo pudesse entender a informação. A separação em treino e teste foi feita com estratificação para preservar a proporção da variável alvo em ambos os conjuntos.

In [ ]:
corr = df[num_cols + ['inadimplente']].corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de correlação')
plt.show()

corr_target = corr['inadimplente'].sort_values(ascending=False)
print(corr_target.round(3).to_string())

### Como interpretar os resultados

Ao executar essas métricas, o objetivo é avaliar se o modelo realmente aprendeu algo útil e não apenas acertou por sorte. A acurácia mostra o percentual geral de acerto, enquanto o ROC-AUC mede a capacidade do modelo de diferenciar clientes mais e menos arriscados.

A matriz de confusão e o relatório de classificação ajudam a entender melhor onde o modelo erra, especialmente em relação aos casos de inadimplência, que normalmente são o foco principal da análise.

### Entendendo quais variáveis mais influenciam

Além de analisar o desempenho, é importante olhar também quais variáveis tiveram mais peso na decisão do modelo. Isso é uma parte muito boa da regressão logística, porque ela permite interpretar o comportamento do classificador.

Se variáveis como renda, score, quantidade de dívidas e a razão renda/valor solicitado aparecerem com valor relevante, isso reforça a hipótese de que o risco de crédito está muito ligado à capacidade financeira do cliente.

## 3. Limpeza e preparação dos dados

Agora vou verificar se a base está em condições de ser usada no modelo. A qualidade dos dados é fundamental, porque erros na preparação podem distorcer o resultado mesmo quando o modelo estiver correto.

Nesse passo, eu procuro duplicatas, valores faltantes, dados fora do padrão e possíveis inconsistências. Se não houver problema relevante, eu deixo claro na análise, porque isso também faz parte do processo e mostra que a base está adequada para a etapa seguinte.

In [ ]:
print('Duplicatas:', df.duplicated().sum())
print('\nValores ausentes por coluna:\n', df.isna().sum())

for col in ['idade', 'tempo_emprego_anos', 'score_credito', 'prazo_meses', 'dividas_ativas']:
    q1 = df[col].quantile(0.01)
    q99 = df[col].quantile(0.99)
    print(f'{col}: faixa aproximada [{q1:.2f}, {q99:.2f}]')

## 4. Engenharia de atributos

Além das variáveis originais, eu também posso criar indicadores derivados que tragam uma visão mais útil para o problema. Em crédito, muitas vezes a combinação entre variáveis explica melhor o risco do que cada variável isolada.

Uma variável que faz bastante sentido aqui é a relação entre renda e valor solicitado. Quando essa relação fica baixa, isso indica que o valor pedido está alto em comparação com a capacidade de pagamento do cliente, o que costuma sinalizar maior risco de inadimplência.

Essa é uma boa oportunidade de mostrar que a análise vai além da base bruta e busca transformar informações em sinais de negócio.

In [ ]:
df['renda_sobre_valor'] = df['renda_mensal'] / df['valor_solicitado']
df['valor_por_prazo'] = df['valor_solicitado'] / df['prazo_meses']

df[['renda_sobre_valor', 'valor_por_prazo', 'inadimplente']].head()

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x='inadimplente', y='renda_sobre_valor')
plt.title('Renda sobre valor solicitado por inadimplência')
plt.show()

print(df.groupby('inadimplente')['renda_sobre_valor'].mean().round(4).to_string())

## 5. Preparação para modelagem

A partir deste ponto, a base já foi entendida e a qualidade foi avaliada. Agora vou preparar os dados para a modelagem, separando treino e teste e ajustando as variáveis para que o algoritmo consiga aprender corretamente.

O principal cuidado aqui é garantir que o modelo seja treinado em um conjunto e avaliado em outro, para evitar um resultado otimista gerado por memorizar os dados de treino. Esse é um conceito essencial em Ciência de Dados e vale ser explicado em uma apresentação.

In [ ]:
target = 'inadimplente'
features = [col for col in df.columns if col != target]
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

num_cols = X.select_dtypes(include=['number']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=500, random_state=42))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

## 6. Modelo: regressão logística

Depois da preparação dos dados, chega o momento de treinar o modelo. Eu escolhi a regressão logística por ser simples, fácil de interpretar e muito usada em problemas de classificação binária.

O modelo vai aprender a relação entre as variáveis de entrada e a chance de uma proposta cair em inadimplência. A vantagem é que, além de prever, ele também nos mostra quais fatores influenciam mais o resultado.

## 7. Avaliação e interpretação

Agora vou avaliar o desempenho do modelo em dados que ele ainda não viu. Esse passo é essencial porque a ideia não é apenas treinar um algoritmo, mas verificar se ele consegue generalizar para situações novas.

Como referência simples, vou comparar o modelo com a estratégia mais básica possível: prever a classe majoritária do conjunto de treino para todos os registros de teste.

In [ ]:
# Referência simples: classificador que prevê sempre a classe majoritária
majority_class = y_train.mode().iloc[0]
baseline_pred = np.full(len(y_test), majority_class)
baseline_acc = accuracy_score(y_test, baseline_pred)

print('Acurácia da referência simples:', round(baseline_acc, 4))
print('Acurácia do modelo:', round(accuracy_score(y_test, y_pred), 4))
print('ROC-AUC do modelo:', round(roc_auc_score(y_test, y_proba), 4))
print('\nRelatório de classificação:\n')
print(classification_report(y_test, y_pred, digits=4))
print('\nMatriz de confusão:\n', confusion_matrix(y_test, y_pred))

In [ ]:
coef = model.named_steps['model'].coef_[0]
feature_names = model.named_steps['preprocessor'].get_feature_names_out()

coef_df = pd.DataFrame({'feature': feature_names, 'coeficiente': coef})
coef_df.sort_values('coeficiente', ascending=False).head(10)

## 8. Conclusão

Ao longo da análise, ficou claro que a inadimplência está mais ligada a fatores financeiros do que a outros aspectos isolados, como idade ou tempo de trabalho. As variáveis que mais chamam atenção foram renda, score de crédito, quantidade de dívidas ativas e a relação entre renda e valor solicitado.

Esses resultados fazem sentido para o contexto do problema, porque uma proposta de crédito com valor alto em relação à renda do cliente e com score baixo tende a apresentar maior risco. Isso é um padrão muito comum em análise de crédito e ajuda a embasar a decisão da empresa.

O modelo de regressão logística conseguiu separar melhor os perfis mais arriscados dos menos arriscados do que uma referência simples baseada na classe majoritária. Mesmo assim, ele também tem limites. Em uma situação real, seria importante combinar esse tipo de análise com regras de negócio, revisão manual em casos críticos e acompanhamento contínuo dos resultados.

Em resumo, a análise mostrou que a instituição pode usar dados históricos para identificar padrões importantes de risco, mas deve sempre interpretar os resultados com cuidado e considerar o contexto do negócio antes de tomar decisões automatizadas.